# Colab 전차 3D Gaussian Splatting 학습

이 노트북은 **Google Colab**에서 전차별 3D 모델을 생성합니다.

## ⚠️ 사전 설정
- **런타임 → 런타임 유형 변경 → GPU** 선택 (T4 이상 권장)
- Google Drive 공유 문서함 또는 MyDrive에 `전차데이터.zip` 업로드

## 흐름
1. Drive 마운트 & 전차데이터 로드
2. 각도별 scene → 전차별 scene 병합
3. COLMAP + 3D Gaussian Splatting 설치 (vocab tree 포함)
4. 전차별 COLMAP → 3D GS 학습 (**전체 데이터** 사용, vocab_tree_matcher)
5. 결과 Drive 저장

## 1. Drive 마운트 & 전차데이터 압축 해제

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
# 전차데이터.zip 위치 (아래 순서로 탐색)
from pathlib import Path
drive = Path("/content/drive")
tanks_base = Path("/content/data/tanks")
# 이미 압축 해제됐는지 확인 (3. 라벨링 존재)
already_extracted = (
    (tanks_base / "전차데이터" / "3. 라벨링").exists()
    or (tanks_base / "전차데이터" / "전차데이터" / "3. 라벨링").exists()
    or (tanks_base / "3. 라벨링").exists()
    or (tanks_base.exists() and any(tanks_base.rglob("3. 라벨링")))
)

if already_extracted:
    print("✓ 이미 압축 해제됨. /content/data/tanks 사용")
    get_ipython().system('find /content/data/tanks -maxdepth 5 -type d -name "*라벨*" 2>/dev/null')
else:
    candidates = [
        drive / "Shareddrives/최종_데이터/전차데이터.zip",
        drive / "MyDrive/데이터 분석과정/한화_최종프로젝트/최종_데이터/전차데이터.zip",
        drive / "MyDrive/hanhwa_final/전차데이터.zip",
    ]
    ZIP_PATH = None
    for p in candidates:
        if p.exists():
            ZIP_PATH = str(p)
            print(f"발견: {ZIP_PATH}")
            break
    if not ZIP_PATH:
        print("⚠️ 전차데이터.zip을 찾을 수 없습니다. 아래 경로 중 하나에 업로드하세요:")
        for p in candidates:
            print(f"  - {p.parent}")
        get_ipython().system('ls -la /content/drive/MyDrive/ 2>/dev/null || true')
    else:
        get_ipython().system('mkdir -p /content/data/tanks')
        get_ipython().system(f'unzip -q "{ZIP_PATH}" -d "/content/data/tanks"')
        print("압축 해제 완료. 라벨링 폴더:")
        get_ipython().system('find /content/data/tanks -maxdepth 5 -type d -name "*라벨*" 2>/dev/null')

✓ 이미 압축 해제됨. /content/data/tanks 사용
/content/data/tanks/전차데이터/1. 동영상 관련/동영상 라벨링
/content/data/tanks/전차데이터/1. 동영상 관련/동영상 라벨링/k1a1/동영상 라벨링
/content/data/tanks/전차데이터/1. 동영상 관련/동영상 라벨링/tiger/동영상 라벨링
/content/data/tanks/전차데이터/1. 동영상 관련/동영상 라벨링/t-90a/동영상 라벨링
/content/data/tanks/전차데이터/1. 동영상 관련/동영상 라벨링/m1a2/동영상 라벨링
/content/data/tanks/전차데이터/1. 동영상 관련/동영상 라벨링/k2/동영상 라벨링
/content/data/tanks/전차데이터/1. 동영상 관련/동영상 라벨링/90식/동영상 라벨링
/content/data/tanks/전차데이터/3. 라벨링
/content/data/tanks/전차데이터/3. 라벨링/tiger/라벨링
/content/data/tanks/전차데이터/3. 라벨링/M1A2/라벨링
/content/data/tanks/전차데이터/3. 라벨링/K2/라벨링
/content/data/tanks/전차데이터/3. 라벨링/K1A1/라벨링
/content/data/tanks/전차데이터/3. 라벨링/90식/라벨링
/content/data/tanks/전차데이터/3. 라벨링/T-90a/라벨링


## 2. 각도별 scene 생성 (18개)

In [22]:
from pathlib import Path
import shutil

def is_image(p: Path) -> bool:
    return p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}

def prepare_3d_scenes(src_root: Path, out_root: Path) -> None:
    """실제 구조: tank/라벨링/pose/서브폴더(0도,45도 등)/*.jpg"""
    count_scenes = 0
    count_images = 0
    for tank_dir in src_root.iterdir():
        if not tank_dir.is_dir():
            continue
        tank_name = tank_dir.name
        mid_dirs = [d for d in tank_dir.iterdir() if d.is_dir()]
        if len(mid_dirs) == 1 and "라벨" in mid_dirs[0].name:
            label_root = mid_dirs[0]
        else:
            label_root = tank_dir
        for pose_dir in label_root.iterdir():
            if not pose_dir.is_dir():
                continue
            pose_name = pose_dir.name
            scene_name = f"{tank_name}_{pose_name}"
            out_images = out_root / scene_name / "images"
            out_images.mkdir(parents=True, exist_ok=True)
            copied_here = 0
            seen_names = set()
            for p in pose_dir.rglob("*"):
                if is_image(p):
                    rel = p.relative_to(pose_dir)
                    # 서브폴더(0도,45도 등)마다 동일 파일명 → 고유 이름
                    dst_name = f"{rel.parts[0]}_{p.name}" if len(rel.parts) > 1 else p.name
                    if dst_name in seen_names:
                        dst_name = f"{rel.parts[0]}_{p.stem}_{len(seen_names)}{p.suffix}"
                    seen_names.add(dst_name)
                    shutil.copy2(p, out_images / dst_name)
                    copied_here += 1
                    count_images += 1
            if copied_here > 0:
                count_scenes += 1
                print(f"[scene] {scene_name}: {copied_here} images")
    print(f"\n총 scene: {count_scenes}, 이미지: {count_images}")

# 라벨링 루트 탐색 (zip: 전차데이터/3.라벨링, 로컬: 전차데이터/전차데이터/3.라벨링)
def find_labeling_root(base: Path) -> Path | None:
    for cand in [
        base / "전차데이터" / "3. 라벨링",           # zip 압축 해제 직후
        base / "전차데이터" / "전차데이터" / "3. 라벨링",
        base / "3. 라벨링",
    ]:
        if cand.exists():
            return cand
    for d in base.rglob("3. 라벨링"):
        if d.is_dir():
            return d
    return None

LABELING = find_labeling_root(Path("/content/data/tanks"))
if LABELING is None:
    print("❌ '3. 라벨링' 폴더를 찾을 수 없습니다. 위 셀(압축 해제)을 먼저 실행하세요.")
    import subprocess
    subprocess.run("find /content/data/tanks -maxdepth 5 -type d 2>/dev/null | head -30", shell=True)
else:
    print(f"라벨링 경로: {LABELING}")
    SCENES_ROOT = Path("/content/data/3d_scenes")
    SCENES_ROOT.mkdir(parents=True, exist_ok=True)
    prepare_3d_scenes(LABELING, SCENES_ROOT)

라벨링 경로: /content/data/tanks/전차데이터/3. 라벨링
[scene] tiger_1촬영30도: 1098 images
[scene] tiger_1촬영60도: 1106 images
[scene] tiger_1촬영90도: 1105 images
[scene] M1A2_M1A2_90도각도_포신: 5523 images
[scene] M1A2_M1A2_45도각도_포신: 5605 images
[scene] M1A2_M1A2_정면각도_포신: 5612 images
[scene] K2_K2_45도각도_포신: 5335 images
[scene] K2_K2_정면도각도_포신: 5400 images
[scene] K2_K2_90도각도_포신: 5441 images
[scene] K1A1_K1A1_45도각도: 5664 images
[scene] K1A1_K1A1_정면각도: 5394 images
[scene] K1A1_K1A1_90도각도: 5526 images
[scene] 90식_90식_정면각도-포신: 5340 images
[scene] 90식_90식_90도각도_포신: 5416 images
[scene] 90식_90식_45도각도_포신: 5301 images
[scene] T-90a_T-90A_90도각도_포신: 5610 images
[scene] T-90a_T-90A_45도각도_포신: 5381 images
[scene] T-90a_T-90A_정면각도_포신: 5369 images

총 scene: 18, 이미지: 85226


## 3. 전차별 scene 병합 (6개)

In [23]:
def prepare_tank_merged_scenes(scenes_root: Path, out_root: Path) -> None:
    tank_to_scenes = {}
    for scene_dir in scenes_root.iterdir():
        if not scene_dir.is_dir():
            continue
        images_dir = scene_dir / "images"
        if not images_dir.exists():
            continue
        tank_name = scene_dir.name.split("_")[0]
        if tank_name not in tank_to_scenes:
            tank_to_scenes[tank_name] = []
        tank_to_scenes[tank_name].append(scene_dir)

    total_images = 0
    for tank_name, scene_dirs in tank_to_scenes.items():
        out_images = out_root / tank_name / "images"
        out_images.mkdir(parents=True, exist_ok=True)
        copied = 0
        seen = set()
        for scene_dir in scene_dirs:
            for p in (scene_dir / "images").rglob("*"):
                if not is_image(p):
                    continue
                dst_name = f"{p.stem}{p.suffix}"
                if dst_name in seen:
                    dst_name = f"{scene_dir.name}_{p.stem}{p.suffix}"
                seen.add(dst_name)
                shutil.copy2(p, out_images / dst_name)
                copied += 1
                total_images += 1
        print(f"[전차] {tank_name}: {copied} images (from {len(scene_dirs)} scenes)")
    print(f"\n총 6개 전차, {total_images} images")

TANK_MERGED_ROOT = Path("/content/data/3d_scenes_by_tank")
TANK_MERGED_ROOT.mkdir(parents=True, exist_ok=True)
prepare_tank_merged_scenes(SCENES_ROOT, TANK_MERGED_ROOT)

[전차] tiger: 3309 images (from 3 scenes)
[전차] K2: 16176 images (from 4 scenes)
[전차] 90식: 16057 images (from 4 scenes)
[전차] M1A2: 16740 images (from 4 scenes)
[전차] K1A1: 16584 images (from 4 scenes)
[전차] T-90a: 16360 images (from 4 scenes)

총 6개 전차, 85226 images


## 3.5 이미지 서브샘플링 (선택 — 빠른 학습용)

**전체 데이터**로 학습하려면 이 셀을 **실행하지 마세요**.  
빠른 테스트만 할 때만 실행 (전차당 80장으로 축소).

In [24]:
# 전체 데이터 사용 시 이 셀 건너뛰기. 빠른 테스트만 할 때만 실행.
USE_SUBSAMPLING = False  # True로 바꾸면 전차당 80장만 사용
MAX_IMAGES_PER_TANK = 80

def subsample_tank_images(tank_root: Path, max_images: int) -> None:
    img_dir = tank_root / "images"
    if not img_dir.exists():
        return
    images = sorted([p for p in img_dir.iterdir() if is_image(p)])
    n = len(images)
    if n <= max_images:
        print(f"  {tank_root.name}: {n}장 유지")
        return
    step = n / max_images
    indices = [int(i * step) for i in range(max_images)]
    keep = [images[i] for i in indices]
    remove = [p for p in images if p not in keep]
    for p in remove:
        p.unlink()
    print(f"  {tank_root.name}: {n} → {len(keep)}장 (삭제 {len(remove)}장)")

if USE_SUBSAMPLING:
    for tank_dir in TANK_MERGED_ROOT.iterdir():
        if tank_dir.is_dir():
            subsample_tank_images(tank_dir, MAX_IMAGES_PER_TANK)
else:
    print("전체 데이터 사용 (서브샘플링 건너뜀)")

전체 데이터 사용 (서브샘플링 건너뜀)


In [25]:
# 데이터 구조/샘플 이미지 검증
from pathlib import Path
from PIL import Image

BASE = Path("/content/data/tanks")
SCENES_BY_TANK = Path("/content/data/3d_scenes_by_tank")

print("[1] 전차데이터 핵심 디렉토리 존재 여부")
for p in [
    BASE,
    BASE / "전차데이터",
    BASE / "전차데이터" / "3. 라벨링",
    SCENES_BY_TANK,
]:
    print(f"- {p}:", "OK" if p.exists() else "MISSING")

print("\n[2] 전차별 이미지 수 집계")
if SCENES_BY_TANK.exists():
    tanks = sorted([d for d in SCENES_BY_TANK.iterdir() if d.is_dir()])
    for t in tanks:
        img_dir = t / "images"
        if not img_dir.exists():
            print(f"- {t.name}: images 폴더 없음")
            continue
        imgs = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}])
        print(f"- {t.name}: {len(imgs)}장")

    print("\n[3] 전차별 샘플 이미지 3장 메타정보")
    for t in tanks:
        img_dir = t / "images"
        if not img_dir.exists():
            continue
        imgs = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}])
        if not imgs:
            print(f"\n[{t.name}] 샘플 없음")
            continue
        print(f"\n[{t.name}] 샘플")
        for p in imgs[:3]:
            try:
                with Image.open(p) as im:
                    print(f"  - {p.name}: {im.width}x{im.height}, mode={im.mode}")
            except Exception as e:
                print(f"  - {p.name}: 읽기 실패 ({e})")
else:
    print("/content/data/3d_scenes_by_tank 가 없습니다. Cell 7까지 먼저 실행하세요.")

[1] 전차데이터 핵심 디렉토리 존재 여부
- /content/data/tanks: OK
- /content/data/tanks/전차데이터: OK
- /content/data/tanks/전차데이터/3. 라벨링: OK
- /content/data/3d_scenes_by_tank: OK

[2] 전차별 이미지 수 집계
- 90식: 16057장
- K1A1: 16584장
- K2: 16176장
- M1A2: 16740장
- T-90a: 16360장
- tiger: 3309장
- tiger_quicktest: 80장

[3] 전차별 샘플 이미지 3장 메타정보

[90식] 샘플
  - 90식_45도각도_포신0도_90식_45도각도_포신0도0000.jpg: 416x312, mode=RGB
  - 90식_45도각도_포신0도_90식_45도각도_포신0도0001.jpg: 416x312, mode=RGB
  - 90식_45도각도_포신0도_90식_45도각도_포신0도0002.jpg: 416x312, mode=RGB

[K1A1] 샘플
  - K1A1_45도각도_포신0도_K1A1_45도각도_포신0도0000.jpg: 416x312, mode=RGB
  - K1A1_45도각도_포신0도_K1A1_45도각도_포신0도0001.jpg: 416x312, mode=RGB
  - K1A1_45도각도_포신0도_K1A1_45도각도_포신0도0002.jpg: 416x312, mode=RGB

[K2] 샘플
  - K2_45도각도_포신45도_K2_45도각도_포신45도0000.jpg: 416x312, mode=RGB
  - K2_45도각도_포신45도_K2_45도각도_포신45도0001.jpg: 416x312, mode=RGB
  - K2_45도각도_포신45도_K2_45도각도_포신45도0002.jpg: 416x312, mode=RGB

[M1A2] 샘플
  - M1A2_45도각도_포신0도_M1A2_45도각도_포신0도0000.jpg: 416x312, mode=RGB
  - M1A2_45도각도_포신0도_M1A2_45도각

## 4. COLMAP + 3D Gaussian Splatting 설치

### COLMAP Vocabulary Tree (대량 이미지용)

1만 장 이상 이미지는 `exhaustive_matcher` 대신 `vocab_tree_matcher` 사용.  
1M words 트리: 1만~10만 장 규모에 적합.

In [26]:
# COLMAP vocab tree (1만~10만 장용)
VOCAB_TREE_URL = "https://github.com/colmap/colmap/releases/download/3.11.1/vocab_tree_flickr100K_words1M.bin"
VOCAB_TREE_PATH = "/content/vocab_tree_flickr100K_words1M.bin"
from pathlib import Path
if not Path(VOCAB_TREE_PATH).exists():
    import urllib.request
    urllib.request.urlretrieve(VOCAB_TREE_URL, VOCAB_TREE_PATH)
print("Vocab tree:", VOCAB_TREE_PATH)

Vocab tree: /content/vocab_tree_flickr100K_words1M.bin


In [27]:
!sudo apt-get update -y
!sudo apt-get install -y colmap
!colmap -h | head -3

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease                         
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease             
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                   
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat

In [34]:
%cd /content

# GPU 런타임 확인 (학습은 CUDA 필수)
import torch, subprocess, shutil
print(f"PyTorch: {torch.__version__}, torch.version.cuda={torch.version.cuda}, cuda_available={torch.cuda.is_available()}")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("[WARN] nvidia-smi 명령을 찾지 못했습니다. (환경에 따라 미설치 가능)")
if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임이 아닙니다. Colab에서 '런타임 유형 변경 -> GPU' 후 재시작하세요.")

# 이전 실행 잔여물 제거
!rm -rf gaussian-splatting

# gsplat 백엔드 포크 사용 (CUDA 확장 빌드 최소화)
!git clone https://github.com/liruilong940607/gaussian-splatting.git
%cd gaussian-splatting
!pip install -U pip setuptools wheel -q
!pip install plyfile tqdm gsplat scipy -q

/content
PyTorch: 2.10.0+cu128, torch.version.cuda=12.8, cuda_available=True
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 928, done.
remote: Total 928 (delta 0), reused 0 (delta 0), pack-reused 928 (from 1)
Receiving objects: 100% (928/928), 78.68 MiB | 33.07 MiB/s, done.
Resolving deltas: 100% (513/513), done.
/content/gaussian-splatting


In [35]:
# simple_knn 대체: scipy KDTree로 distCUDA2 구현 (CUDA 컴파일 불필요)
# liruilong940607 포크는 gsplat 사용 → diff-gaussian-rasterization/simple-knn 빌드 불필요
import os
gm_path = "/content/gaussian-splatting/scene/gaussian_model.py"
with open(gm_path, "r") as f:
    content = f.read()

# simple_knn 대신 scipy 기반 distCUDA2 사용
old_import = "from simple_knn._C import distCUDA2"
new_import = '''# distCUDA2: simple_knn 대체 (Colab CUDA 12.4 호환)
try:
    from simple_knn._C import distCUDA2
except ImportError:
    from scipy.spatial import KDTree
    import torch
    def distCUDA2(points):
        pts = points.detach().cpu().float().numpy()
        dists, _ = KDTree(pts).query(pts, k=4)
        mean_dists = (dists[:, 1:] ** 2).mean(1)
        return torch.tensor(mean_dists, dtype=points.dtype, device=points.device)'''

if old_import in content and "scipy.spatial" not in content:
    content = content.replace(old_import, new_import)
    with open(gm_path, "w") as f:
        f.write(content)
    print("✓ gaussian_model.py: scipy 기반 distCUDA2 fallback 적용")

# train.py shape 호환 패치 (gsplat 출력 텐서 차이 대응)
train_path = "/content/gaussian-splatting/train.py"
with open(train_path, "r") as f:
    tcontent = f.read()

old_block = """                gaussians.max_radii2D[visibility_filter] = torch.max(gaussians.max_radii2D[visibility_filter], radii[visibility_filter])
                gaussians.add_densification_stats(viewspace_point_tensor, visibility_filter, image.shape[2], image.shape[1])"""
new_block = """                # gsplat 버전/옵션에 따라 visibility_filter/radii shape이 달라질 수 있어 길이를 안전하게 정렬
                vf = visibility_filter
                rd = radii
                if vf.dim() > 1:
                    vf = vf.any(dim=-1)
                if rd.dim() > 1:
                    rd = rd.max(dim=-1).values
                vf = vf.reshape(-1).bool()
                rd = rd.reshape(-1)

                N = gaussians.max_radii2D.shape[0]
                M = min(N, vf.numel(), rd.numel())
                vf_full = torch.zeros(N, dtype=torch.bool, device=gaussians.max_radii2D.device)
                rd_full = torch.zeros(N, dtype=rd.dtype, device=rd.device)
                if M > 0:
                    vf_full[:M] = vf[:M]
                    rd_full[:M] = rd[:M]

                gaussians.max_radii2D[vf_full] = torch.max(gaussians.max_radii2D[vf_full], rd_full[vf_full])
                gaussians.add_densification_stats(viewspace_point_tensor, vf_full, image.shape[2], image.shape[1])"""

old_block_v1 = """                # gsplat 버전/옵션에 따라 visibility_filter, radii shape이 [1,N] 또는 [N]로 달라질 수 있음
                visibility_filter = visibility_filter.reshape(-1).bool()
                radii = radii.reshape(-1)
                gaussians.max_radii2D[visibility_filter] = torch.max(gaussians.max_radii2D[visibility_filter], radii[visibility_filter])
                gaussians.add_densification_stats(viewspace_point_tensor, visibility_filter, image.shape[2], image.shape[1])"""

if old_block in tcontent:
    tcontent = tcontent.replace(old_block, new_block)
    with open(train_path, "w") as f:
        f.write(tcontent)
    print("✓ train.py: visibility_filter/radii shape 안전 정렬 패치 적용")
elif old_block_v1 in tcontent:
    tcontent = tcontent.replace(old_block_v1, new_block)
    with open(train_path, "w") as f:
        f.write(tcontent)
    print("✓ train.py: 기존 shape 패치를 안전 정렬 버전으로 업그레이드")
elif "vf_full" in tcontent:
    print("✓ train.py: visibility_filter/radii shape 안전 정렬 패치 이미 적용됨")
else:
    print("[WARN] train.py 패치 대상 블록을 찾지 못했습니다. train.py 변경 여부를 확인하세요.")

# 설치 확인 (gsplat + 패치)
import subprocess
r = subprocess.run(["python", "-c", "import gsplat; from scene.gaussian_model import GaussianModel; print('✓ gsplat + gaussian_model 준비 완료')"],
                   cwd="/content/gaussian-splatting", capture_output=True, text=True)
if r.returncode != 0:
    print("⚠️ 확인 실패:", r.stderr or r.stdout)
else:
    print(r.stdout)

✓ gaussian_model.py: scipy 기반 distCUDA2 fallback 적용
✓ train.py: visibility_filter/radii shape 안전 정렬 패치 적용
✓ gsplat + gaussian_model 준비 완료



## 5. 전차별 3D 모델 학습 (COLMAP → 3D GS)

### 빠른 검증 모드 (QUICK_TEST)

- **QUICK_TEST=True**: tiger만 80장으로 테스트 (~10~20분). 파이프라인 검증용.
- **QUICK_TEST=False**: 전체 6개 전차, 8만+ 이미지. 성공 확인 후 변경.

In [ ]:
import os
import re
import shutil
import subprocess
import unicodedata
from pathlib import Path

# Colab headless 환경: COLMAP Qt/OpenGL 오류 방지
os.environ["QT_QPA_PLATFORM"] = "offscreen"
os.environ["XDG_RUNTIME_DIR"] = "/tmp/runtime-root"

# --- 실행 전 환경 점검 (GPU + CUDA PyTorch 필수) ---
import torch, shutil
print(f"PyTorch: {torch.__version__}, torch.version.cuda={torch.version.cuda}, cuda_available={torch.cuda.is_available()}")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("[WARN] nvidia-smi 명령을 찾지 못했습니다. (환경에 따라 미설치 가능)")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU를 사용할 수 없습니다. Colab에서 '런타임 유형 변경 -> GPU'로 바꾼 뒤 런타임 재시작 후 다시 실행하세요. "
        "(현재는 CPU용 PyTorch 또는 CPU 런타임 상태)"
    )

# 빠른 검증: True면 tiger만 80장으로 테스트 (약 10~20분). False면 전체 6개 전차.
QUICK_TEST = False  # 성공 확인 후 False로 변경해 전체 학습
MAX_QUICK_IMAGES = 80

DATA_ROOT = Path("/content/data/3d_scenes_by_tank")
GS_ROOT = Path("/content/gaussian-splatting")

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"데이터 폴더가 없습니다: {DATA_ROOT}. Cell 5~7을 먼저 실행하세요.")
if not GS_ROOT.exists():
    raise FileNotFoundError(f"gaussian-splatting 폴더가 없습니다: {GS_ROOT}. Cell 14~15를 먼저 실행하세요.")


def _safe_ascii_name(name: str) -> str:
    stem, ext = os.path.splitext(name)
    norm = unicodedata.normalize("NFKD", stem)
    ascii_stem = norm.encode("ascii", "ignore").decode("ascii")
    ascii_stem = re.sub(r"[^A-Za-z0-9._-]+", "_", ascii_stem).strip("._-")
    if not ascii_stem:
        ascii_stem = "img"
    return f"{ascii_stem}{ext.lower()}"


def sanitize_image_filenames(img_dir: Path) -> int:
    """COLMAP binary 파서 UTF-8 문제 방지를 위해 파일명을 ASCII로 정규화."""
    files = sorted([p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])
    used = set()
    renamed = 0
    for i, p in enumerate(files):
        base = _safe_ascii_name(p.name)
        stem, ext = os.path.splitext(base)
        candidate = f"{i:06d}_{stem}{ext}"
        k = 1
        while candidate in used or (img_dir / candidate).exists() and (img_dir / candidate) != p:
            candidate = f"{i:06d}_{stem}_{k}{ext}"
            k += 1
        used.add(candidate)
        dst = img_dir / candidate
        if p.name != candidate:
            p.rename(dst)
            renamed += 1
    return renamed


def run_colmap_and_train(scene_name: str) -> bool:
    # 셀 재실행 누락/런타임 변경 상황에서도 매번 GPU 상태 확인
    if not torch.cuda.is_available():
        raise RuntimeError("학습 중 CUDA가 비활성화되었습니다. 런타임(GPU) 상태를 다시 확인하세요.")

    scene_root = DATA_ROOT / scene_name
    img_dir = scene_root / "images"
    if not img_dir.exists():
        print(f"[SKIP] {scene_name}: images 폴더 없음")
        return False

    renamed = sanitize_image_filenames(img_dir)
    if renamed:
        print(f"[{scene_name}] 파일명 ASCII 정규화: {renamed}개")

    n_imgs = len([p for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])
    if n_imgs == 0:
        print(f"[SKIP] {scene_name}: 이미지 0장")
        return False

    # 이전 실행 잔여물 제거
    db_path = scene_root / "database.db"
    if db_path.exists():
        db_path.unlink()
    sparse_root = scene_root / "sparse"
    if sparse_root.exists():
        import shutil
        shutil.rmtree(sparse_root)
    os.makedirs(str(sparse_root), exist_ok=True)

    sparse_dir = scene_root / "sparse" / "0"  # mapper가 생성

    sparse_str = str(sparse_root.resolve())
    sparse_dir_str = str(sparse_dir)
    assert sparse_root.is_dir(), f"sparse 디렉터리 생성 실패: {sparse_root}"

    # COLMAP (대량 이미지: 해상도·특징 수 제한으로 메모리 절약)
    # Colab headless: OpenGL 없음 → SiftExtraction/SiftMatching CPU 모드 강제
    env = dict(os.environ)
    print(f"\n=== [{scene_name}] COLMAP ({n_imgs}장) ===")
    result = subprocess.run([
        "colmap", "feature_extractor",
        "--database_path", str(db_path),
        "--image_path", str(img_dir),
        "--ImageReader.camera_model", "PINHOLE",
        "--ImageReader.single_camera", "1",
        "--SiftExtraction.use_gpu", "0",
    ], capture_output=True, text=True, env=env)
    if result.returncode != 0:
        print("COLMAP stderr:", (result.stderr or "")[-2000:])
        raise subprocess.CalledProcessError(result.returncode, result.args)

    # 대량 이미지: vocab_tree_matcher (exhaustive는 1만 장 이상 시 메모리 부족)
    vocab_path = "/content/vocab_tree_flickr100K_words1M.bin"
    for cmd_name, args in [
        ("vocab_tree_matcher", ["colmap", "vocab_tree_matcher", "--database_path", str(db_path),
         "--VocabTreeMatching.vocab_tree_path", vocab_path, "--SiftMatching.use_gpu", "0"]),
        ("mapper", ["colmap", "mapper", "--database_path", str(scene_root / "database.db"),
         "--image_path", str(scene_root / "images"), "--output_path", sparse_str]),
        ("model_converter", ["colmap", "model_converter", "--input_path", sparse_dir_str,
         "--output_path", sparse_dir_str, "--output_type", "TXT"]),
    ]:
        r = subprocess.run(args, capture_output=True, text=True, env=env)
        if r.returncode != 0:
            print(f"COLMAP {cmd_name} stderr:", (r.stderr or "")[-2000:])
            raise subprocess.CalledProcessError(r.returncode, args)

    # sparse/0 결과 검증 (cameras.txt, images.txt 필수)
    if not (sparse_dir / "cameras.txt").exists() or not (sparse_dir / "images.txt").exists():
        raise RuntimeError(f"COLMAP mapper 출력 없음: {sparse_dir} (cameras.txt/images.txt 확인)")

    # 3D Gaussian Splatting (gsplat 백엔드 포크 사용, CUDA 컴파일 불필요)
    print(f"=== [{scene_name}] 3D Gaussian Splatting (gsplat) ===")
    train_cmd = ["python", "train.py", "-s", str(scene_root), "-m", f"output/{scene_name}"]
    r = subprocess.run(train_cmd, cwd=str(GS_ROOT), env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print("train.py stderr:", (r.stderr or "")[-3000:])
        raise subprocess.CalledProcessError(r.returncode, train_cmd)

    return True


# 학습할 전차 목록
tanks = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir() and (d / "images").exists()])
if not tanks:
    raise RuntimeError("학습 가능한 전차 폴더가 없습니다. Cell 7 결과를 확인하세요.")

if QUICK_TEST:
    src_tank = "tiger" if "tiger" in tanks else tanks[0]

    # 원본 유지: tiger_quicktest 폴더에 80장만 복사 (재실행 안정성 위해 폴더 초기화)
    import shutil
    quick_dir = DATA_ROOT / "tiger_quicktest"
    if quick_dir.exists():
        shutil.rmtree(quick_dir)
    (quick_dir / "images").mkdir(parents=True, exist_ok=True)

    imgs = sorted([p for p in (DATA_ROOT / src_tank / "images").iterdir()
                   if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])
    if not imgs:
        raise RuntimeError(f"{src_tank} 이미지가 없습니다.")

    step = max(1.0, len(imgs) / float(MAX_QUICK_IMAGES))
    for i in range(min(MAX_QUICK_IMAGES, len(imgs))):
        src = imgs[min(int(i * step), len(imgs) - 1)]
        dst = quick_dir / "images" / f"{i:04d}_{src.name}"
        shutil.copy2(src, dst)

    tanks = ["tiger_quicktest"]
    print(f"[QUICK_TEST] {src_tank} → tiger_quicktest: {min(MAX_QUICK_IMAGES, len(imgs))}장 (원본 유지)")

print(f"학습 대상: {tanks}" + (" (빠른 검증 모드)" if QUICK_TEST else ""))

PyTorch: 2.10.0+cu128, torch.version.cuda=12.8, cuda_available=True
[QUICK_TEST] tiger → tiger_quicktest: 80장 (원본 유지)
학습 대상: ['tiger_quicktest'] (빠른 검증 모드)


In [37]:
# 전차별 순차 학습 (각 전차당 약 6~50분 소요)
import traceback

results = []
for tank in tanks:
    try:
        ok = run_colmap_and_train(tank)
        results.append((tank, "OK" if ok else "SKIP"))
    except Exception as e:
        print(f"[ERROR] {tank}: {e}")
        traceback.print_exc(limit=2)
        results.append((tank, "ERROR"))

print("\n학습 결과 요약")
for name, status in results:
    print(f"- {name}: {status}")

print("\n전차별 3D 모델 학습 완료")

[tiger_quicktest] 파일명 ASCII 정규화: 80개

=== [tiger_quicktest] COLMAP (80장) ===
=== [tiger_quicktest] 3D Gaussian Splatting (gsplat) ===

학습 결과 요약
- tiger_quicktest: OK

전차별 3D 모델 학습 완료


## 6. 결과 확인

In [38]:
!find /content/gaussian-splatting/output -name "point_cloud.ply" 2>/dev/null

/content/gaussian-splatting/output/tiger_quicktest/point_cloud/iteration_7000/point_cloud.ply
/content/gaussian-splatting/output/tiger_quicktest/point_cloud/iteration_30000/point_cloud.ply


## 7. Drive로 결과 저장

In [39]:
import shutil

OUT_DIR = Path("/content/gaussian-splatting/output")
DRIVE_SAVE = Path("/content/drive/MyDrive/hanhwa_final/3d_models_by_tank")

if not Path("/content/drive/MyDrive").exists():
    print("Drive 마운트가 안 된 상태입니다. Cell 2를 다시 실행하세요.")
elif not OUT_DIR.exists():
    print("학습 결과 폴더가 없습니다:", OUT_DIR)
else:
    DRIVE_SAVE.parent.mkdir(parents=True, exist_ok=True)
    if DRIVE_SAVE.exists():
        shutil.rmtree(DRIVE_SAVE)
    shutil.copytree(OUT_DIR, DRIVE_SAVE)
    print("Drive 저장 완료:", DRIVE_SAVE)

Drive 저장 완료: /content/drive/MyDrive/hanhwa_final/3d_models_by_tank
